In [49]:
import os
import re
import gc
import time
import warnings
from typing import Optional

import numpy as np
import pandas as pd

from arch import arch_model
from joblib import Parallel, delayed
from scipy.stats import skew, kurtosis
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA
from pathlib import Path

warnings.filterwarnings("ignore")


In [50]:
# =============================================================================
# WINDOW CONFIG
# =============================================================================


INPUT_START = 0
INPUT_END = 480

TARGET_START = 480
TARGET_END = 600

INPUT_LEN = INPUT_END - INPUT_START
TARGET_LEN = TARGET_END - TARGET_START
TOTAL_LEN = max(INPUT_END, TARGET_END)

FORECAST_OFFSET = TARGET_START - INPUT_END
FORECAST_HORIZON = TARGET_END - INPUT_END

WINDOW_TAG = (
    f"in{INPUT_START}_{INPUT_END}"
    f"_target{TARGET_START}_{TARGET_END}"
)

# =============================================================================
# CONFIG
# =============================================================================

EPSILON = 1e-5
METRIC_EPSILON = 1e-12
RET_CLIP = 0.05

N_CLUSTERS = 3
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SEED = 42

SPEC = dict(ar=1, ma=1, p=1, q=1, dist="normal")

KNN_KS = [3, 5, 10, 20, 40, 80]
CV_FOLDS = 5

ARIMA_MAXITER = 50
GARCH_MAXITER = 50

CLUSTER_METHOD = "csv_cluster_level"
CACHE_DIR = "arma_garch_cluster_knn_cache_csv_clusters"

FORCE_REBUILD_CLUSTER_FEATURES = True

# Set this to a lower num for a quicker test run. None for the everything.
DEV_N_TIME_IDS: Optional[int] = None

N_JOBS = 8

In [51]:
# =============================================================================
# CLUSTER CSV PATH
# =============================================================================

def get_arma_garch_dir() -> Path:
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd().resolve()


ARMA_GARCH_DIR = get_arma_garch_dir()
PROJECT_ROOT = ARMA_GARCH_DIR.parent

CLUSTER_CSV_DIR = (
    PROJECT_ROOT
    / "Clustering+Feature engineering"
    / "processed"
    / "clustering"
)

CLUSTER_CSV_FILENAME = "best_cluster_labels.csv"

CLUSTER_CSV_PATH = None


def resolve_cluster_csv_path() -> Path:
    if CLUSTER_CSV_PATH is not None:
        path = Path(CLUSTER_CSV_PATH).expanduser().resolve()

        if not path.exists():
            raise FileNotFoundError(f"Cluster CSV not found: {path}")

        return path

    if CLUSTER_CSV_FILENAME is not None:
        path = (CLUSTER_CSV_DIR / CLUSTER_CSV_FILENAME).resolve()

        if not path.exists():
            raise FileNotFoundError(f"Cluster CSV not found: {path}")

        return path

    if not CLUSTER_CSV_DIR.exists():
        raise FileNotFoundError(
            f"Cluster folder not found: {CLUSTER_CSV_DIR}"
        )

    csv_files = sorted(CLUSTER_CSV_DIR.glob("*.csv"))

    if len(csv_files) == 0:
        raise FileNotFoundError(
            f"No CSV files found in cluster folder: {CLUSTER_CSV_DIR}"
        )


    return csv_files[0].resolve()

In [52]:
# =============================================================================
# HELPERS
# =============================================================================

def _safe_log_return_matrix(mat: np.ndarray) -> np.ndarray:
    safe = np.where(mat > 0, mat, 1e-6).astype(np.float32)

    ret = np.zeros_like(safe)
    ret[1:] = np.log(safe[1:] / safe[:-1])

    np.clip(ret, -RET_CLIP, RET_CLIP, out=ret)

    np.nan_to_num(
        ret,
        copy=False,
        nan=0.0,
        posinf=RET_CLIP,
        neginf=-RET_CLIP,
    )

    return ret


def robust_fill(mat: np.ndarray) -> np.ndarray:
    bad = ~np.isfinite(mat) | (mat <= 0)

    if not bad.any():
        return mat

    out = mat.copy()
    rows = out.shape[0]
    row_idx = np.arange(rows)[:, None]

    fwd = np.where(~bad, row_idx, -1)
    np.maximum.accumulate(fwd, axis=0, out=fwd)

    rev = np.where(~bad, row_idx, rows)
    bwd = rows - 1 - np.maximum.accumulate(
        (rows - 1 - rev)[::-1],
        axis=0,
    )[::-1]

    use = np.where(fwd >= 0, fwd, bwd)
    all_bad = (use < 0) | (use >= rows)
    use = np.where(all_bad, 0, use)

    out = out[use, np.arange(out.shape[1])]

    return np.where(all_bad, 1.0, out)


def safe_skew(x: np.ndarray) -> float:
    if np.std(x) <= 1e-12:
        return 0.0

    val = skew(x, bias=False)

    if not np.isfinite(val):
        return 0.0

    return float(val)


def safe_kurt(x: np.ndarray) -> float:
    if np.std(x) <= 1e-12:
        return 0.0

    val = kurtosis(x, bias=False, fisher=True)

    if not np.isfinite(val):
        return 0.0

    return float(val)


def safe_autocorr1(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)

    if len(x) < 3:
        return 0.0

    a = x[:-1]
    b = x[1:]

    if np.std(a) <= 1e-12 or np.std(b) <= 1e-12:
        return 0.0

    val = np.corrcoef(a, b)[0, 1]

    if not np.isfinite(val):
        return 0.0

    return float(val)


def parse_stock_id_from_col(col) -> int:
    if isinstance(col, (int, np.integer)):
        return int(col)

    s = str(col)

    if s.isdigit():
        return int(s)

    nums = re.findall(r"\d+", s)

    if len(nums) == 0:
        raise ValueError(f"Could not parse stock_id from column name: {col}")

    return int(nums[-1])


def _rv_metrics(rv_pred: np.ndarray, rv_act: np.ndarray) -> dict:
    fp = np.maximum(np.asarray(rv_pred).ravel(), METRIC_EPSILON)
    fa = np.maximum(np.asarray(rv_act).ravel(), METRIC_EPSILON)

    mse_val = mean_squared_error(fa, fp)
    err = fp - fa
    pct = err / fa
    ratio = fa / (fp + METRIC_EPSILON)

    try:
        r2_val = float(r2_score(fa, fp))
    except Exception:
        r2_val = np.nan

    return {
        "mse": float(mse_val),
        "rmse": float(np.sqrt(mse_val)),
        "r2": r2_val,
        "mae": float(np.mean(np.abs(err))),
        "rmspe": float(np.sqrt(np.mean(pct ** 2))),
        "mape": float(np.mean(np.abs(pct))),
        "qlike": float(np.mean(ratio - np.log(ratio + METRIC_EPSILON) - 1.0)),
    }


def metric_pack(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return _rv_metrics(rv_pred=y_pred, rv_act=y_true)


def _print_metrics(m: dict, label: str) -> None:
    print(f"\n  {label}")
    print(f"    MSE   : {m['mse']:.10f}")
    print(f"    RMSE  : {m['rmse']:.10f}")
    print(f"    R2    : {m['r2']:.6f}")
    print(f"    MAE   : {m['mae']:.10f}")
    print(f"    RMSPE : {m['rmspe'] * 100.0:.4f}%")
    print(f"    MAPE  : {m['mape'] * 100.0:.4f}%")
    print(f"    QLIKE : {m['qlike']:.10f}")

In [53]:
# =============================================================================
# SPLITS AND CLUSTERING
# =============================================================================

def make_splits(
    df: pd.DataFrame,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
) -> dict:
    rng = np.random.default_rng(seed)

    tids = df["time_id"].unique().copy()
    rng.shuffle(tids)

    n = len(tids)
    n_test = int(n * test_ratio)
    n_val = int(n * val_ratio)

    return {
        "split_summary": {
            "train": {"ids": tids[n_test + n_val:]},
            "val": {"ids": tids[n_test:n_test + n_val]},
            "test": {"ids": tids[:n_test]},
        }
    }

def build_cluster_map_from_csv(stock_cols: list) -> pd.Series:
    cluster_csv_path = resolve_cluster_csv_path()

    print("Loading stock clusters from CSV")
    print(f"  Cluster CSV: {cluster_csv_path}")

    cluster_df = pd.read_csv(cluster_csv_path)

    required_cols = {"stock_id", "cluster_id"}

    if not required_cols.issubset(cluster_df.columns):
        raise ValueError(
            f"Cluster CSV must contain columns {required_cols}. "
            f"Found columns: {list(cluster_df.columns)}"
        )

    cluster_df = cluster_df[["stock_id", "cluster_id"]].copy()
    cluster_df["stock_id"] = cluster_df["stock_id"].astype(int)
    cluster_df["cluster_id"] = cluster_df["cluster_id"].astype(int)

    if cluster_df["stock_id"].duplicated().any():
        dupes = cluster_df.loc[
            cluster_df["stock_id"].duplicated(),
            "stock_id",
        ].tolist()

        raise ValueError(f"Duplicate stock_id values in cluster CSV: {dupes}")

    cluster_ids = sorted(cluster_df["cluster_id"].unique().tolist())
    expected_ids = list(range(len(cluster_ids)))

    if cluster_ids != expected_ids:
        raise ValueError(
            "Cluster IDs must be contiguous and start at 0. "
            f"Found cluster IDs: {cluster_ids}. Expected: {expected_ids}"
        )

    stock_id_to_cluster = dict(
        zip(cluster_df["stock_id"], cluster_df["cluster_id"])
    )

    labels = []
    missing_from_csv = []

    for col in stock_cols:
        stock_id = parse_stock_id_from_col(col)

        if stock_id not in stock_id_to_cluster:
            missing_from_csv.append(stock_id)
            continue

        labels.append(stock_id_to_cluster[stock_id])

    if missing_from_csv:
        raise ValueError(
            "These stock columns were found in the data but are missing "
            f"from the cluster CSV: {sorted(missing_from_csv)}"
        )

    stock_ids_in_data = {
        parse_stock_id_from_col(c)
        for c in stock_cols
    }

    unused_csv_ids = sorted(
        set(stock_id_to_cluster.keys()) - stock_ids_in_data
    )

    if unused_csv_ids:
        print(
            "Warning: these stock IDs are in the cluster CSV but not in "
            f"the data, so they will be ignored: {unused_csv_ids}"
        )

    cluster_map = pd.Series(
        labels,
        index=stock_cols,
        name="cluster_id",
        dtype=np.int32,
    )

    print("Using CSV stock clusters")
    print("  Cluster sizes:", cluster_map.value_counts().sort_index().to_dict())

    return cluster_map


def get_or_build_cluster_map(
    stock_cols: list,
    n_clusters: int,
    seed: int,
) -> pd.Series:
    os.makedirs(CACHE_DIR, exist_ok=True)

    cluster_path = os.path.join(
        CACHE_DIR,
        f"cluster_map_{CLUSTER_METHOD}_{WINDOW_TAG}_seed{seed}_k{n_clusters}.pkl",
    )

    if os.path.exists(cluster_path):
        cluster_map = pd.read_pickle(cluster_path)

        if int(cluster_map.nunique()) != int(n_clusters):
            raise ValueError(
                f"Cached cluster map has {cluster_map.nunique()} clusters, "
                f"but N_CLUSTERS={n_clusters}. Delete cache or fix N_CLUSTERS."
            )

        print("Loaded cached CSV cluster map")
        print("  Cluster sizes:", cluster_map.value_counts().sort_index().to_dict())
        return cluster_map

    cluster_map = build_cluster_map_from_csv(stock_cols)

    if int(cluster_map.nunique()) != int(n_clusters):
        raise ValueError(
            f"CSV has {cluster_map.nunique()} clusters, "
            f"but N_CLUSTERS={n_clusters}. Change N_CLUSTERS."
        )

    cluster_map.to_pickle(cluster_path)

    return cluster_map

In [54]:
# =============================================================================
# CLUSTER ARMA-GARCH FEATURE EXTRACTION
# =============================================================================
def make_cluster_feature_names(n_clusters: int) -> list[str]:
    names = [
        "log_rv_input",
        "log_rv_last_window",
        "log_rv_scaled",
        "rv_last_window_to_input",
        "rv_scaled_to_input",
        "cluster_mean_return",
        "cluster_std_return",
        "cluster_mean_abs_return",
        "cluster_skew_return",
        "cluster_kurt_return",
        "cluster_max_abs_return",
        "cluster_autocorr1",
        "cluster_size_scaled",
    ]

    names.extend([f"cluster_onehot_{i}" for i in range(n_clusters)])

    names.extend(
        [
            "garch_forecast_rv",
            "garch_forecast_step_vol",
            "garch_forecast_to_input",
            "cond_vol_last",
            "cond_vol_mean",
            "cond_vol_std",
            "omega",
            "alpha_1",
            "beta_1",
            "alpha_plus_beta",
            "arma_ar_1",
            "arma_ma_1",
            "arma_sigma2",
            "resid_std",
            "fit_ok",
        ]
    )

    return names


def get_arma_param(fit, name: str, default: float = 0.0) -> float:
    try:
        names = list(getattr(fit, "param_names", []))
        vals = np.asarray(fit.params, dtype=np.float64)

        if name in names:
            idx = names.index(name)
            val = float(vals[idx])

            if np.isfinite(val):
                return val

    except Exception:
        pass

    return default


def cluster_base_feature_vector(task: dict) -> list[float]:
    r = np.asarray(task["inp_ret"], dtype=np.float64)

    onehot = [0.0] * task["n_clusters"]
    onehot[int(task["cluster_id"])] = 1.0

    return [
        float(np.log(task["rv_in"] + EPSILON)),
        float(np.log(task["rv_last120"] + EPSILON)),
        float(np.log(task["rv_scaled"] + EPSILON)),
        float(task["rv_last120"] / (task["rv_in"] + EPSILON)),
        float(task["rv_scaled"] / (task["rv_in"] + EPSILON)),
        float(np.mean(r)),
        float(np.std(r)),
        float(np.mean(np.abs(r))),
        safe_skew(r),
        safe_kurt(r),
        float(np.max(np.abs(r))),
        safe_autocorr1(r),
        float(task["cluster_size"] / max(task["n_stocks"], 1)),
    ] + onehot


def extract_cluster_armagarch_feature_row(task: dict, spec: dict) -> dict:
    scale = 100.0

    inp_ret = np.asarray(task["inp_ret"], dtype=np.float64)
    base = cluster_base_feature_vector(task)

    fallback_rv = float(task["rv_scaled"])
    fallback_step_vol = fallback_rv / np.sqrt(TARGET_LEN)

    fit_features = [
        fallback_rv,
        fallback_step_vol,
        fallback_rv / (task["rv_in"] + EPSILON),
        fallback_step_vol,
        fallback_step_vol,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        float(np.var(inp_ret * scale)),
        float(np.std(inp_ret * scale)),
        0.0,
    ]

    fit_ok = 0.0

    try:
        if len(inp_ret) < INPUT_LEN:
            raise ValueError("short input")

        if not np.isfinite(inp_ret).all():
            raise ValueError("bad input")

        if np.std(inp_ret) <= 1e-12:
            raise ValueError("flat input")

        y = inp_ret * scale

        try:
            arma_fit = ARIMA(
                y,
                order=(spec["ar"], 0, spec["ma"]),
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(
                method_kwargs={"maxiter": ARIMA_MAXITER},
            )

        except TypeError:
            arma_fit = ARIMA(
                y,
                order=(spec["ar"], 0, spec["ma"]),
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit()

        resid = np.asarray(arma_fit.resid, dtype=np.float64)

        burn = max(spec["ar"], spec["ma"], 1)
        resid = resid[burn:]
        resid = resid[np.isfinite(resid)]

        if len(resid) < 50:
            raise ValueError("short residual")

        resid = resid - np.mean(resid)

        if np.std(resid) <= 1e-12:
            raise ValueError("flat residual")

        garch = arch_model(
            resid,
            mean="Zero",
            vol="Garch",
            p=spec["p"],
            q=spec["q"],
            dist=spec["dist"],
            rescale=False,
        )

        garch_fit = garch.fit(
            disp="off",
            show_warning=False,
            update_freq=0,
            tol=1e-4,
            options={"maxiter": GARCH_MAXITER},
        )

        fc = garch_fit.forecast(
            horizon=FORECAST_HORIZON,
            reindex=False,
        )

        var_fc = np.asarray(fc.variance.values.flatten(), dtype=np.float64)
        var_fc = var_fc[np.isfinite(var_fc)]
        var_fc = np.maximum(var_fc, 0.0)

        if len(var_fc) < FORECAST_HORIZON:
            raise ValueError("bad forecast length")

        target_var_fc = var_fc[
            FORECAST_OFFSET:FORECAST_OFFSET + TARGET_LEN
        ]

        if len(target_var_fc) != TARGET_LEN:
            raise ValueError("bad target forecast window")

        garch_forecast_rv = float(np.sqrt(np.sum(target_var_fc))) / scale
        garch_forecast_step_vol = float(np.sqrt(np.mean(target_var_fc))) / scale

        cond_vol = np.asarray(garch_fit.conditional_volatility, dtype=np.float64)
        cond_vol = cond_vol[np.isfinite(cond_vol)] / scale

        if len(cond_vol) == 0:
            raise ValueError("bad conditional vol")

        params = garch_fit.params

        omega = float(params.get("omega", 0.0))
        alpha_1 = float(params.get("alpha[1]", 0.0))
        beta_1 = float(params.get("beta[1]", 0.0))

        arma_ar_1 = get_arma_param(arma_fit, "ar.L1", 0.0)
        arma_ma_1 = get_arma_param(arma_fit, "ma.L1", 0.0)
        arma_sigma2 = get_arma_param(arma_fit, "sigma2", float(np.var(resid)))

        fit_features = [
            garch_forecast_rv,
            garch_forecast_step_vol,
            garch_forecast_rv / (task["rv_in"] + EPSILON),
            float(cond_vol[-1]),
            float(np.mean(cond_vol)),
            float(np.std(cond_vol)),
            omega,
            alpha_1,
            beta_1,
            alpha_1 + beta_1,
            arma_ar_1,
            arma_ma_1,
            arma_sigma2,
            float(np.std(resid)),
            1.0,
        ]

        fit_ok = 1.0

    except Exception:
        pass

    x = np.array(base + fit_features, dtype=np.float32)

    np.nan_to_num(
        x,
        copy=False,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    return {
        "x": x,
        "y_log": float(task["y_log"]),
        "rv_in": float(task["rv_in"]),
        "rv_fut": float(task["rv_fut"]),
        "rv_last120": float(task["rv_last120"]),
        "rv_scaled": float(task["rv_scaled"]),
        "time_id": int(task["time_id"]),
        "cluster_id": int(task["cluster_id"]),
        "cluster_size": int(task["cluster_size"]),
        "fit_ok": fit_ok,
    }

def build_cluster_feature_tasks(
    df: pd.DataFrame,
    stock_cols: list,
    cluster_map: pd.Series,
    time_ids: np.ndarray,
    split_name: str,
    dev_n_time_ids: Optional[int] = None,
) -> list[dict]:
    if dev_n_time_ids is not None:
        time_ids = time_ids[:dev_n_time_ids]

    cluster_ids = sorted(cluster_map.unique())
    n_clusters = len(cluster_ids)
    n_stocks = len(stock_cols)

    stock_to_pos = {s: i for i, s in enumerate(stock_cols)}

    cluster_stock_idx = {
        int(cid): np.array(
            [
                stock_to_pos[stock]
                for stock in cluster_map.index[cluster_map == cid]
            ],
            dtype=np.int32,
        )
        for cid in cluster_ids
    }

    cluster_sizes = {
        int(cid): len(cluster_stock_idx[int(cid)])
        for cid in cluster_ids
    }

    df_sub = df[df["time_id"].isin(time_ids)].sort_values(
        ["time_id", "seconds_in_bucket"],
    )

    tasks = []

    for tid, grp in df_sub.groupby("time_id", sort=True):
        mat_raw = grp[stock_cols].values.astype(np.float32)

        if mat_raw.shape[0] < TOTAL_LEN:
            continue

        mat_in = robust_fill(mat_raw[INPUT_START:INPUT_END])
        ret_in_mat = _safe_log_return_matrix(mat_in)

        target_context_start = max(TARGET_START - 1, 0)
        mat_target_context = robust_fill(
            mat_raw[target_context_start:TARGET_END]
        )
        ret_target_context = _safe_log_return_matrix(mat_target_context)

        target_offset = TARGET_START - target_context_start
        ret_fut_mat = ret_target_context[
            target_offset:target_offset + TARGET_LEN
        ]

        if ret_fut_mat.shape[0] != TARGET_LEN:
            continue

        for cid in cluster_ids:
            cid_int = int(cid)
            idx = cluster_stock_idx[cid_int]

            r_in_stocks = ret_in_mat[:, idx]
            r_fut_stocks = ret_fut_mat[:, idx]

            inp_ret = r_in_stocks.mean(axis=1).astype(np.float64)

            rv_in = float(
                np.sqrt(np.einsum("ij,ij->j", r_in_stocks, r_in_stocks)).mean()
            ) + EPSILON

            rv_fut = float(
                np.sqrt(np.einsum("ij,ij->j", r_fut_stocks, r_fut_stocks)).mean()
            ) + EPSILON

            r_last = r_in_stocks[-TARGET_LEN:]

            rv_last120 = float(
                np.sqrt(np.einsum("ij,ij->j", r_last, r_last)).mean()
            ) + EPSILON

            rv_scaled = float(rv_in * np.sqrt(TARGET_LEN / INPUT_LEN)) + EPSILON
            y_log = float(np.log(rv_fut / rv_in))

            tasks.append(
                {
                    "time_id": int(tid),
                    "cluster_id": cid_int,
                    "cluster_size": int(cluster_sizes[cid_int]),
                    "n_clusters": int(n_clusters),
                    "n_stocks": int(n_stocks),
                    "inp_ret": inp_ret,
                    "rv_in": rv_in,
                    "rv_fut": rv_fut,
                    "rv_last120": rv_last120,
                    "rv_scaled": rv_scaled,
                    "y_log": y_log,
                }
            )

    print(
        f"  {split_name}: built {len(tasks):,} cluster tasks "
        f"from {len(set(t['time_id'] for t in tasks)):,} time_ids"
    )

    return tasks

def cluster_feature_cache_path(split_name: str, spec: dict) -> str:
    dev_tag = "full" if DEV_N_TIME_IDS is None else f"dev{DEV_N_TIME_IDS}"

    return os.path.join(
        CACHE_DIR,
        (
            f"features_{split_name}_{dev_tag}"
            f"_{WINDOW_TAG}"
            f"_clusterlevel"
            f"_clusterstyle_{CLUSTER_METHOD}"
            f"_k{N_CLUSTERS}"
            f"_seed{SEED}"
            f"_arma{spec['ar']}{spec['ma']}"
            f"_garch{spec['p']}{spec['q']}"
            f"_{spec['dist']}.npz"
        ),
    )


def build_or_load_cluster_features(
    df: pd.DataFrame,
    stock_cols: list,
    cluster_map: pd.Series,
    time_ids: np.ndarray,
    split_name: str,
    spec: dict,
    n_jobs: int,
) -> dict:
    os.makedirs(CACHE_DIR, exist_ok=True)

    path = cluster_feature_cache_path(split_name, spec)
    feature_names = make_cluster_feature_names(N_CLUSTERS)

    if os.path.exists(path) and not FORCE_REBUILD_CLUSTER_FEATURES:
        print(f"Loading cached cluster-level {split_name} features from {path}")

        z = np.load(path, allow_pickle=True)

        return {
            "X": z["X"],
            "y_log": z["y_log"],
            "rv_in": z["rv_in"],
            "rv_fut": z["rv_fut"],
            "rv_last120": z["rv_last120"],
            "rv_scaled": z["rv_scaled"],
            "time_id": z["time_id"],
            "cluster_id": z["cluster_id"],
            "cluster_size": z["cluster_size"],
            "fit_ok": z["fit_ok"],
            "feature_names": list(z["feature_names"]),
        }

    tasks = build_cluster_feature_tasks(
        df=df,
        stock_cols=stock_cols,
        cluster_map=cluster_map,
        time_ids=time_ids,
        split_name=split_name,
        dev_n_time_ids=DEV_N_TIME_IDS,
    )

    print(
        f"  {split_name}: fitting cluster-level ARMA-GARCH features "
        f"with n_jobs={n_jobs}"
    )

    start = time.perf_counter()

    if n_jobs == 1:
        rows = [
            extract_cluster_armagarch_feature_row(task, spec)
            for task in tasks
        ]

    else:
        rows = Parallel(
            n_jobs=n_jobs,
            backend="loky",
            verbose=5,
            batch_size=16,
        )(
            delayed(extract_cluster_armagarch_feature_row)(task, spec)
            for task in tasks
        )

    elapsed = (time.perf_counter() - start) / 60.0

    X = np.vstack([r["x"] for r in rows]).astype(np.float32)
    y_log = np.array([r["y_log"] for r in rows], dtype=np.float32)
    rv_in = np.array([r["rv_in"] for r in rows], dtype=np.float32)
    rv_fut = np.array([r["rv_fut"] for r in rows], dtype=np.float32)
    rv_last120 = np.array([r["rv_last120"] for r in rows], dtype=np.float32)
    rv_scaled = np.array([r["rv_scaled"] for r in rows], dtype=np.float32)
    time_id = np.array([r["time_id"] for r in rows], dtype=np.int32)
    cluster_id = np.array([r["cluster_id"] for r in rows], dtype=np.int32)
    cluster_size = np.array([r["cluster_size"] for r in rows], dtype=np.int32)
    fit_ok = np.array([r["fit_ok"] for r in rows], dtype=np.float32)

    print(
        f"  {split_name}: cluster-level X={X.shape}, "
        f"fit success={fit_ok.mean() * 100:.1f}%, "
        f"time={elapsed:.2f} min"
    )

    np.savez_compressed(
        path,
        X=X,
        y_log=y_log,
        rv_in=rv_in,
        rv_fut=rv_fut,
        rv_last120=rv_last120,
        rv_scaled=rv_scaled,
        time_id=time_id,
        cluster_id=cluster_id,
        cluster_size=cluster_size,
        fit_ok=fit_ok,
        feature_names=np.array(feature_names, dtype=object),
    )

    print(f"  Saved cluster-level {split_name} features to {path}")

    return {
        "X": X,
        "y_log": y_log,
        "rv_in": rv_in,
        "rv_fut": rv_fut,
        "rv_last120": rv_last120,
        "rv_scaled": rv_scaled,
        "time_id": time_id,
        "cluster_id": cluster_id,
        "cluster_size": cluster_size,
        "fit_ok": fit_ok,
        "feature_names": feature_names,
    }

In [55]:

# =============================================================================
# KNN
# =============================================================================

def fit_knn(
    X_train: np.ndarray,
    y_train: np.ndarray,
    k: int,
) -> tuple[StandardScaler, KNeighborsRegressor]:
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_train)

    model = KNeighborsRegressor(
        n_neighbors=min(k, len(X_train)),
        weights="distance",
        metric="minkowski",
        p=2,
        algorithm="brute",
        n_jobs=-1,
    )

    model.fit(Xs, y_train)

    return scaler, model


def predict_knn_log(
    scaler: StandardScaler,
    model: KNeighborsRegressor,
    X: np.ndarray,
) -> np.ndarray:
    Xs = scaler.transform(X)
    pred = model.predict(Xs).astype(np.float64)
    return np.clip(pred, -5.0, 5.0)


def cross_validate_knn_per_cluster(
    train_data: dict,
    knn_ks: list[int],
    cv_folds: int = 5,
) -> tuple[dict[int, int], pd.DataFrame]:
    print("KNN cross validation, cluster-level target")

    rows = []
    best_k_by_cluster = {}

    for cid in sorted(np.unique(train_data["cluster_id"])):
        mask = train_data["cluster_id"] == cid

        Xc = train_data["X"][mask]
        yc = train_data["y_log"][mask]
        rvin = train_data["rv_in"][mask]
        rvfut = train_data["rv_fut"][mask]
        groups = train_data["time_id"][mask]

        unique_groups = np.unique(groups)
        n_splits = min(cv_folds, len(unique_groups))
        gkf = GroupKFold(n_splits=n_splits)

        print(
            f"\n  Cluster {int(cid)} | "
            f"rows={len(yc):,} | "
            f"stocks={int(train_data['cluster_size'][mask][0])} | "
            f"time_ids={len(unique_groups):,}"
        )

        cluster_rows = []

        for k in knn_ks:
            fold_metrics = []

            for tr_idx, va_idx in gkf.split(Xc, yc, groups):
                scaler, model = fit_knn(
                    X_train=Xc[tr_idx],
                    y_train=yc[tr_idx],
                    k=min(k, len(tr_idx)),
                )

                log_pred = predict_knn_log(
                    scaler=scaler,
                    model=model,
                    X=Xc[va_idx],
                )

                rv_pred = np.exp(log_pred) * rvin[va_idx]
                fold_metrics.append(metric_pack(rvfut[va_idx], rv_pred))

            row = {
                "cluster_id": int(cid),
                "k": int(k),
                "cv_mse_mean": float(np.mean([m["mse"] for m in fold_metrics])),
                "cv_rmse_mean": float(np.mean([m["rmse"] for m in fold_metrics])),
                "cv_r2_mean": float(np.nanmean([m["r2"] for m in fold_metrics])),
                "cv_mae_mean": float(np.mean([m["mae"] for m in fold_metrics])),
                "cv_rmspe_mean": float(np.mean([m["rmspe"] for m in fold_metrics])),
                "cv_rmspe_std": float(np.std([m["rmspe"] for m in fold_metrics])),
                "cv_mape_mean": float(np.mean([m["mape"] for m in fold_metrics])),
                "cv_qlike_mean": float(np.mean([m["qlike"] for m in fold_metrics])),
            }

            cluster_rows.append(row)

            print(
                f"    k={k:<4} | "
                f"RMSPE={row['cv_rmspe_mean'] * 100.0:.4f}% "
                f"+/- {row['cv_rmspe_std'] * 100.0:.4f}% | "
                f"MSE={row['cv_mse_mean']:.10f} | "
                f"RMSE={row['cv_rmse_mean']:.10f} | "
                f"MAE={row['cv_mae_mean']:.10f} | "
                f"MAPE={row['cv_mape_mean'] * 100.0:.4f}% | "
                f"QLIKE={row['cv_qlike_mean']:.10f} | "
                f"R2={row['cv_r2_mean']:.6f}"
            )

        cluster_df = pd.DataFrame(cluster_rows).sort_values("cv_rmspe_mean")
        best_k = int(cluster_df.iloc[0]["k"])
        best_k_by_cluster[int(cid)] = best_k

        print(f"    Best k for cluster {int(cid)}: {best_k}")

        rows.extend(cluster_rows)

    cv_df = pd.DataFrame(rows)

    print("\nBest k by cluster:")
    print(best_k_by_cluster)

    return best_k_by_cluster, cv_df


def fit_predict_per_cluster(
    train_data: dict,
    eval_data: dict,
    best_k_by_cluster: dict[int, int],
) -> tuple[np.ndarray, dict]:
    log_pred = np.zeros(len(eval_data["y_log"]), dtype=np.float64)
    fitted = {}

    for cid in sorted(np.unique(eval_data["cluster_id"])):
        train_mask = train_data["cluster_id"] == cid
        eval_mask = eval_data["cluster_id"] == cid

        k = int(best_k_by_cluster.get(int(cid), 10))

        scaler, model = fit_knn(
            X_train=train_data["X"][train_mask],
            y_train=train_data["y_log"][train_mask],
            k=k,
        )

        log_pred[eval_mask] = predict_knn_log(
            scaler=scaler,
            model=model,
            X=eval_data["X"][eval_mask],
        )

        fitted[int(cid)] = {
            "k": k,
            "scaler": scaler,
            "model": model,
        }

    return np.clip(log_pred, -5.0, 5.0), fitted


def evaluate_cluster_predictions(
    label: str,
    data: dict,
    log_pred: np.ndarray,
) -> dict:
    rv_pred = np.exp(np.clip(log_pred, -5.0, 5.0)) * data["rv_in"]

    raw = metric_pack(data["rv_fut"], rv_pred)
    last = metric_pack(data["rv_fut"], data["rv_last120"])
    scaled = metric_pack(data["rv_fut"], data["rv_scaled"])

    print(f"\n{label}")

    _print_metrics(raw, "Cluster-level raw KNN")
    _print_metrics(last, "Cluster-level last-window baseline")
    _print_metrics(scaled, "Cluster-level scaled-input baseline")

    print("\n  Per-cluster raw KNN scores")

    rows = []

    for cid in sorted(np.unique(data["cluster_id"])):
        mask = data["cluster_id"] == cid

        m = metric_pack(data["rv_fut"][mask], rv_pred[mask])

        rows.append(
            {
                "cluster_id": int(cid),
                "rows": int(mask.sum()),
                "cluster_size": int(data["cluster_size"][mask][0]),
                **m,
            }
        )

    per_cluster = pd.DataFrame(rows)

    for _, row in per_cluster.iterrows():
        print(
            f"    Cluster {int(row['cluster_id']):<3} | "
            f"rows={int(row['rows']):<7} | "
            f"stocks={int(row['cluster_size']):<3} | "
            f"RMSPE={row['rmspe'] * 100.0:.4f}% | "
            f"MSE={row['mse']:.10f} | "
            f"RMSE={row['rmse']:.10f} | "
            f"MAE={row['mae']:.10f} | "
            f"MAPE={row['mape'] * 100.0:.4f}% | "
            f"QLIKE={row['qlike']:.10f} | "
            f"R2={row['r2']:.6f}"
        )

    return {
        "raw": raw,
        "last120": last,
        "scaled": scaled,
        "raw_per_cluster": per_cluster,
        "rv_pred": rv_pred,
    }


def concat_train_val(train_data: dict, val_data: dict) -> dict:
    keys = [
        "X",
        "y_log",
        "rv_in",
        "rv_fut",
        "rv_last120",
        "rv_scaled",
        "time_id",
        "cluster_id",
        "cluster_size",
        "fit_ok",
    ]

    out = {}

    for key in keys:
        out[key] = np.concatenate([train_data[key], val_data[key]])

    out["feature_names"] = train_data["feature_names"]

    return out

In [56]:
# =============================================================================
# FULL PIPELINE
# =============================================================================

def run_knn(
    df: pd.DataFrame,
    n_clusters: int = N_CLUSTERS,
    spec: dict = SPEC,
    val_ratio: float = VAL_RATIO,
    test_ratio: float = TEST_RATIO,
    seed: int = SEED,
    knn_ks: list[int] = KNN_KS,
    cv_folds: int = CV_FOLDS,
    n_jobs: int = N_JOBS,
) -> dict:
    start_all = time.perf_counter()

    print("=" * 80)
    print("Cluster-level KNN with CSV clusters and cluster ARMA-GARCH features")
    print(f"Clustering: CSV k={n_clusters} stock clusters")
    print(
        f"Cluster ARMA({spec['ar']},{spec['ma']})-"
        f"GARCH({spec['p']},{spec['q']}) | "
        f"dist={spec['dist']}"
    )
    print("=" * 80)

    stock_cols = [
        c for c in df.columns
        if c not in ("time_id", "seconds_in_bucket")
    ]

    print(
        f"Rows: {len(df):,} | "
        f"Stocks: {len(stock_cols)} | "
        f"Time IDs: {df['time_id'].nunique():,} | "
        f"Input: {INPUT_START}-{INPUT_END - 1} | "
        f"Target: {TARGET_START}-{TARGET_END - 1} | "
        f"Clusters: {n_clusters} | "
        f"N_JOBS: {n_jobs}"
    )

    t_split = time.perf_counter()

    splits = make_splits(
        df=df,
        val_ratio=val_ratio,
        test_ratio=test_ratio,
        seed=seed,
    )

    split_sec = time.perf_counter() - t_split

    train_ids = splits["split_summary"]["train"]["ids"]
    val_ids = splits["split_summary"]["val"]["ids"]
    test_ids = splits["split_summary"]["test"]["ids"]

    print(
        f"Split: train={len(train_ids):,}, "
        f"val={len(val_ids):,}, "
        f"test={len(test_ids):,}"
    )

    t_cluster = time.perf_counter()

    cluster_map = get_or_build_cluster_map(
        stock_cols=stock_cols,
        n_clusters=n_clusters,
        seed=seed,
    )

    cluster_setup_sec = time.perf_counter() - t_cluster

    expected_cluster_fits = (
        len(train_ids) + len(val_ids) + len(test_ids)
    ) * n_clusters

    if DEV_N_TIME_IDS is not None:
        expected_cluster_fits = (
            min(DEV_N_TIME_IDS, len(train_ids))
            + min(DEV_N_TIME_IDS, len(val_ids))
            + min(DEV_N_TIME_IDS, len(test_ids))
        ) * n_clusters

    print(
        f"Expected cluster ARMA-GARCH fits if cache is missing: "
        f"{expected_cluster_fits:,}"
    )

    t_feat = time.perf_counter()

    cluster_train = build_or_load_cluster_features(
        df, stock_cols, cluster_map, train_ids, "train", spec, n_jobs
    )

    cluster_val = build_or_load_cluster_features(
        df, stock_cols, cluster_map, val_ids, "val", spec, n_jobs
    )

    cluster_test = build_or_load_cluster_features(
        df, stock_cols, cluster_map, test_ids, "test", spec, n_jobs
    )

    feature_extraction_min = (time.perf_counter() - t_feat) / 60.0

    t_cv = time.perf_counter()

    best_k_by_cluster, cv_df = cross_validate_knn_per_cluster(
        cluster_train,
        knn_ks,
        cv_folds,
    )

    cv_min = (time.perf_counter() - t_cv) / 60.0

    print("\nFitting cluster-level KNN on train and evaluating validation")

    t_val = time.perf_counter()

    val_log_pred, val_models = fit_predict_per_cluster(
        cluster_train,
        cluster_val,
        best_k_by_cluster,
    )

    val_predict_sec = time.perf_counter() - t_val

    val_metrics = evaluate_cluster_predictions(
        "Validation evaluation",
        cluster_val,
        val_log_pred,
    )

    print("\nFitting final cluster-level KNN on train plus validation")

    cluster_train_val = concat_train_val(
        cluster_train,
        cluster_val,
    )

    t_test = time.perf_counter()

    test_log_pred, final_models = fit_predict_per_cluster(
        cluster_train_val,
        cluster_test,
        best_k_by_cluster,
    )

    test_predict_sec = time.perf_counter() - t_test

    test_metrics = evaluate_cluster_predictions(
        "Final test evaluation",
        cluster_test,
        test_log_pred,
    )

    total_min = (time.perf_counter() - start_all) / 60.0

    timing_rows = [
        {
            "step": "split data",
            "seconds": split_sec,
            "minutes": split_sec / 60.0,
        },
        {
            "step": "load CSV clusters",
            "seconds": cluster_setup_sec,
            "minutes": cluster_setup_sec / 60.0,
        },
        {
            "step": "feature extraction",
            "seconds": feature_extraction_min * 60.0,
            "minutes": feature_extraction_min,
        },
        {
            "step": "KNN cross validation",
            "seconds": cv_min * 60.0,
            "minutes": cv_min,
        },
        {
            "step": "validation KNN fit plus prediction",
            "seconds": val_predict_sec,
            "minutes": val_predict_sec / 60.0,
        },
        {
            "step": "final KNN fit plus test prediction",
            "seconds": test_predict_sec,
            "minutes": test_predict_sec / 60.0,
        },
        {
            "step": "total pipeline",
            "seconds": total_min * 60.0,
            "minutes": total_min,
        },
    ]

    timing_df = pd.DataFrame(timing_rows)

    os.makedirs(CACHE_DIR, exist_ok=True)

    timing_path = os.path.join(CACHE_DIR, "timing_breakdown_cluster_level.csv")
    cv_path = os.path.join(CACHE_DIR, "knn_cv_results_cluster_level.csv")
    val_cluster_path = os.path.join(CACHE_DIR, "validation_per_cluster_cluster_level.csv")
    test_cluster_path = os.path.join(CACHE_DIR, "test_per_cluster_cluster_level.csv")

    timing_df.to_csv(timing_path, index=False)
    cv_df.to_csv(cv_path, index=False)
    val_metrics["raw_per_cluster"].to_csv(val_cluster_path, index=False)
    test_metrics["raw_per_cluster"].to_csv(test_cluster_path, index=False)

    print("\n" + "=" * 80)
    print("Final summary")
    print("=" * 80)
    print("Model: cluster-level per-cluster KNN")
    print(f"Clustering: {N_CLUSTERS} Stock Clusters")
    print("Features: cluster RV features plus cluster ARMA-GARCH features")
    print("Target: cluster average RV log-ratio")
    print(f"Best k by cluster: {best_k_by_cluster}")

    print("\nValidation, cluster-level")
    _print_metrics(val_metrics["raw"], "Raw KNN")
    print("\nTest, cluster-level")
    _print_metrics(test_metrics["raw"], "Raw KNN")
    print("\nTest baselines, cluster-level")
    _print_metrics(test_metrics["last120"], "Last-window baseline")
    _print_metrics(test_metrics["scaled"], "Scaled-input baseline")

    print("\n" + "=" * 80)
    print("Timing breakdown")
    print("=" * 80)
    print(f"  Split creation                       : {split_sec:.3f} sec")
    print(f"  CSV clusters                   : {cluster_setup_sec / 60.0:.2f} min")
    print(
        f"  Feature extraction                   : "
        f"{feature_extraction_min:.2f} min "
        f"{'[from cache]' if feature_extraction_min < 0.5 else '[computed]'}"
    )
    print(f"  KNN cross validation                 : {cv_min:.2f} min")
    print(f"  Validation KNN fit plus prediction   : {val_predict_sec:.3f} sec")
    print(f"  Final KNN fit plus test prediction   : {test_predict_sec:.3f} sec")
    print(f"  Total pipeline                       : {total_min:.2f} min")

    print("\n  For reporting:")
    print(
        f"    Training time, one-off             : "
        f"{feature_extraction_min + cv_min:.2f} min"
    )
    print(
        f"    Final KNN fit plus test prediction : "
        f"{test_predict_sec:.3f} sec "
        f"({test_predict_sec / max(len(test_ids), 1) * 1000:.1f} ms per auction)"
    )
    print(f"    Timing CSV                         : {timing_path}")
    print(f"    CV CSV                             : {cv_path}")
    print(f"    Validation per-cluster CSV         : {val_cluster_path}")
    print(f"    Test per-cluster CSV               : {test_cluster_path}")
    print("=" * 80)

    return {
        "splits": splits,
        "cluster_map": cluster_map,
        "cluster_train": cluster_train,
        "cluster_val": cluster_val,
        "cluster_test": cluster_test,
        "cv_df": cv_df,
        "best_k_by_cluster": best_k_by_cluster,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "val_models": val_models,
        "final_models": final_models,
        "timing_df": timing_df,
        "timing_path": timing_path,
        "cv_path": cv_path,
        "val_cluster_path": val_cluster_path,
        "test_cluster_path": test_cluster_path,
        "total_minutes": total_min,
        "timings": {
            "split_sec": split_sec,
            "cluster_setup_sec": cluster_setup_sec,
            "feature_extraction_min": feature_extraction_min,
            "cv_min": cv_min,
            "val_predict_sec": val_predict_sec,
            "test_predict_sec": test_predict_sec,
            "total_min": total_min,
        },
    }

In [57]:
# =============================================================================
# LOADING DATA
# =============================================================================

def load_final_csv_chunked(
    path: str = "final.csv",
    chunksize: int = 100_000,
) -> pd.DataFrame:
    header = pd.read_csv(path, nrows=0)

    stock_cols = [
        c for c in header.columns
        if c not in ("time_id", "seconds_in_bucket")
    ]

    dtype_map = {
        "time_id": np.int32,
        "seconds_in_bucket": np.int16,
        **{c: np.float32 for c in stock_cols},
    }

    chunks = []

    print(f"Reading {path} in chunks")

    for i, chunk in enumerate(
        pd.read_csv(
            path,
            dtype=dtype_map,
            chunksize=chunksize,
        )
    ):
        chunks.append(chunk)

        if i % 10 == 0:
            print(f"  chunk {i}, rows so far {(i + 1) * chunksize:,}")

    df = pd.concat(chunks, ignore_index=True)

    print(
        f"Loaded CSV: {df.shape} | "
        f"memory={df.memory_usage(deep=True).sum() / 1e6:.0f} MB"
    )

    return df


def load_final_parquet_batched(
    path: str = "final.parquet",
    batch_size: int = 50_000,
) -> pd.DataFrame:
    import pyarrow.parquet as pq

    print(f"Loading {path} in batches")

    pf = pq.ParquetFile(path)
    chunks = []
    rows_so_far = 0

    for i, batch in enumerate(pf.iter_batches(batch_size=batch_size)):
        chunk = batch.to_pandas()

        if "time_id" in chunk.columns:
            chunk["time_id"] = chunk["time_id"].astype(np.int32)

        if "seconds_in_bucket" in chunk.columns:
            chunk["seconds_in_bucket"] = chunk["seconds_in_bucket"].astype(np.int16)

        stock_cols = [
            c for c in chunk.columns
            if c not in ("time_id", "seconds_in_bucket")
        ]

        for c in stock_cols:
            chunk[c] = chunk[c].astype(np.float32)

        chunks.append(chunk)
        rows_so_far += len(chunk)

        if i % 5 == 0:
            print(f"  batch {i}, rows so far {rows_so_far:,}")

        del batch
        gc.collect()

    df = pd.concat(chunks, ignore_index=True)

    del chunks
    gc.collect()

    print(
        f"Loaded parquet: {df.shape} | "
        f"memory={df.memory_usage(deep=True).sum() / 1e6:.0f} MB"
    )

    return df

In [58]:
# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    if os.path.exists("final.parquet"):
        df = load_final_parquet_batched(
            path="final.parquet",
            batch_size=50_000,
        )

    else:
        print("Loading final.csv and saving final.parquet")

        df = load_final_csv_chunked(
            path="final.csv",
            chunksize=50_000,
        )

        df.to_parquet(
            "final.parquet",
            index=False,
        )

        print("Saved final.parquet")

    results = run_knn(
        df=df,
        n_clusters=N_CLUSTERS,
        spec=SPEC,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        seed=SEED,
        knn_ks=KNN_KS,
        cv_folds=CV_FOLDS,
        n_jobs=N_JOBS,
    )


Loading final.parquet in batches
  batch 0, rows so far 50,000
  batch 5, rows so far 300,000
  batch 10, rows so far 550,000
  batch 15, rows so far 800,000
  batch 20, rows so far 1,050,000
  batch 25, rows so far 1,300,000
  batch 30, rows so far 1,550,000
  batch 35, rows so far 1,800,000
  batch 40, rows so far 2,050,000
  batch 45, rows so far 2,298,000
Loaded parquet: (2298000, 114) | memory=1043 MB
Cluster-level KNN with CSV clusters and cluster ARMA-GARCH features
Clustering: CSV k=3 stock clusters
Cluster ARMA(1,1)-GARCH(1,1) | dist=normal
Rows: 2,298,000 | Stocks: 112 | Time IDs: 3,830 | Input: 0-479 | Target: 480-599 | Clusters: 3 | N_JOBS: 8
Split: train=2,682, val=574, test=574
Loading stock clusters from CSV
  Cluster CSV: C:\Users\jamie\OneDrive\Documents\GitHub\DATA3888-Finance-21\Clustering+Feature engineering\processed\clustering\best_cluster_labels.csv
Using CSV stock clusters
  Cluster sizes: {0: 10, 1: 90, 2: 12}
Expected cluster ARMA-GARCH fits if cache is missin

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  48 tasks      | elapsed:   15.0s
[Parallel(n_jobs=8)]: Done 912 tasks      | elapsed:  1.0min
[Parallel(n_jobs=8)]: Done 2352 tasks      | elapsed:  2.4min
[Parallel(n_jobs=8)]: Done 4368 tasks      | elapsed:  4.9min
[Parallel(n_jobs=8)]: Done 6960 tasks      | elapsed:  7.4min
[Parallel(n_jobs=8)]: Done 8046 out of 8046 | elapsed:  8.7min finished


  train: cluster-level X=(8046, 31), fit success=100.0%, time=8.68 min
  Saved cluster-level train features to arma_garch_cluster_knn_cache_csv_clusters\features_train_full_in0_480_target480_600_clusterlevel_clusterstyle_csv_cluster_level_k3_seed42_arma11_garch11_normal.npz
  val: built 1,722 cluster tasks from 574 time_ids
  val: fitting cluster-level ARMA-GARCH features with n_jobs=8


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  48 tasks      | elapsed:    8.7s
[Parallel(n_jobs=8)]: Done 912 tasks      | elapsed:  1.0min
[Parallel(n_jobs=8)]: Done 1722 out of 1722 | elapsed:  1.8min finished


  val: cluster-level X=(1722, 31), fit success=100.0%, time=1.81 min
  Saved cluster-level val features to arma_garch_cluster_knn_cache_csv_clusters\features_val_full_in0_480_target480_600_clusterlevel_clusterstyle_csv_cluster_level_k3_seed42_arma11_garch11_normal.npz
  test: built 1,722 cluster tasks from 574 time_ids
  test: fitting cluster-level ARMA-GARCH features with n_jobs=8


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  48 tasks      | elapsed:    7.3s
[Parallel(n_jobs=8)]: Done 912 tasks      | elapsed:   56.9s
[Parallel(n_jobs=8)]: Done 1722 out of 1722 | elapsed:  1.8min finished


  test: cluster-level X=(1722, 31), fit success=100.0%, time=1.83 min
  Saved cluster-level test features to arma_garch_cluster_knn_cache_csv_clusters\features_test_full_in0_480_target480_600_clusterlevel_clusterstyle_csv_cluster_level_k3_seed42_arma11_garch11_normal.npz
KNN cross validation, cluster-level target

  Cluster 0 | rows=2,682 | stocks=10 | time_ids=2,682
    k=3    | RMSPE=12.2399% +/- 0.3692% | MSE=0.0000000211 | RMSE=0.0001447484 | MAE=0.0000977236 | MAPE=9.3987% | QLIKE=0.0074993268 | R2=0.965936
    k=5    | RMSPE=11.6156% +/- 0.3877% | MSE=0.0000000204 | RMSE=0.0001423937 | MAE=0.0000941617 | MAPE=8.9504% | QLIKE=0.0067839300 | R2=0.966880
    k=10   | RMSPE=11.1822% +/- 0.5485% | MSE=0.0000000188 | RMSE=0.0001366211 | MAE=0.0000907829 | MAPE=8.5684% | QLIKE=0.0062865333 | R2=0.969621
    k=20   | RMSPE=11.0543% +/- 0.5119% | MSE=0.0000000197 | RMSE=0.0001399170 | MAE=0.0000905813 | MAPE=8.4331% | QLIKE=0.0061400008 | R2=0.968105
    k=40   | RMSPE=11.0614% +/- 0.4724